In [1]:
"""
Week 3: Predictive Analytics & Risk Modeling Pipeline
Construction Project Performance Analytics
"""

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, classification_report, fbeta_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler

# -------------------------------------------------------------------------
# 1. SYNTHETIC DATA GENERATION & FEATURE ENGINEERING
# -------------------------------------------------------------------------
np.random.seed(42)
N = 600

# Base operational attributes
planned_duration = np.random.randint(120, 900, size=N)
planned_budget = np.random.uniform(2.0, 60.0, size=N)  # In Millions USD
subcontractor_reliability = np.clip(np.random.beta(a=6, b=2, size=N), 0.25, 1.0)
weather_delays = np.random.poisson(lam=12, size=N)
safety_incidents = np.random.poisson(lam=1.5, size=N)
material_cost_ratio = np.random.uniform(0.30, 0.65, size=N)

# Engineered Risk Interaction Variables
labor_density = (np.random.normal(loc=6.5, scale=1.5, size=N)).clip(2.0, 12.0)
weather_hazard_ratio = weather_delays / planned_duration
trade_stacking_risk = (labor_density > 8.0).astype(int) * safety_incidents

# Ground truth delay & cost generation with non-linear risk interactions
delay_days = (
    (weather_delays * 1.6)
    + ((1.0 - subcontractor_reliability) * 0.22 * planned_duration)
    + (safety_incidents * 5.5)
    + (trade_stacking_risk * 4.2)
    + np.random.normal(0, 10, size=N)
)
delay_days = np.maximum(0, delay_days)

cost_overrun_pct = (
    (delay_days / planned_duration) * 35.0
    + (material_cost_ratio * 12.0)
    + ((1.0 - subcontractor_reliability) * 15.0)
    + np.random.normal(0, 4, size=N)
)
cost_overrun_pct = np.maximum(-2.0, cost_overrun_pct)

# Binary Target: Severe Failure Risk (Delay >= 60 Days OR Cost Overrun >= 20%)
severe_risk_target = ((delay_days >= 60) | (cost_overrun_pct >= 20.0)).astype(int)

df = pd.DataFrame(
    {
        "Planned_Duration": planned_duration,
        "Planned_Budget_M": planned_budget,
        "Subcontractor_Score": subcontractor_reliability,
        "Weather_Delays": weather_delays,
        "Safety_Incidents": safety_incidents,
        "Material_Cost_Ratio": material_cost_ratio,
        "Labor_Density": labor_density,
        "Weather_Hazard_Ratio": weather_hazard_ratio,
        "Trade_Stacking_Risk": trade_stacking_risk,
        "Delay_Days": delay_days,
        "Cost_Overrun_Pct": cost_overrun_pct,
        "Severe_Risk": severe_risk_target,
    }
)

print(
    f"[INFO] Ingested {len(df)} project records. Positive class prevalence (Severe Risk): {df['Severe_Risk'].mean():.2%}"
)

# -------------------------------------------------------------------------
# 2. MODEL TRAINING & VALIDATION PIPELINE
# -------------------------------------------------------------------------
features = [
    "Planned_Duration",
    "Planned_Budget_M",
    "Subcontractor_Score",
    "Weather_Delays",
    "Safety_Incidents",
    "Material_Cost_Ratio",
    "Labor_Density",
    "Weather_Hazard_Ratio",
    "Trade_Stacking_Risk",
]

X = df[features]
y = df["Severe_Risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Candidate Models
models = {
    "Logistic Regression (ElasticNet)": LogisticRegression(
        penalty="elasticnet", solver="saga", l1_ratio=0.5, max_iter=1000, random_state=42
    ),
    "Random Forest Classifier": RandomForestClassifier(
        n_estimators=150, max_depth=6, min_samples_split=4, random_state=42
    ),
    "Gradient Boosted Trees (GBM)": GradientBoostingClassifier(
        n_estimators=120, learning_rate=0.06, max_depth=4, random_state=42
    ),
}

benchmark_results = []

for name, model in models.items():
    train_data = X_train_scaled if "Logistic" in name else X_train
    test_data = X_test_scaled if "Logistic" in name else X_test

    model.fit(train_data, y_train)
    y_pred = model.predict(test_data)
    y_prob = model.predict_proba(test_data)[:, 1]

    roc_auc = roc_auc_score(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)
    f2 = fbeta_score(y_test, y_pred, beta=2)

    benchmark_results.append(
        {"Model": name, "ROC-AUC": round(roc_auc, 3), "PR-AUC": round(pr_auc, 3), "F2-Score": round(f2, 3)}
    )

print("\n=== CLASSIFICATION BENCHMARK PERFORMANCE ===")
print(pd.DataFrame(benchmark_results).to_string(index=False))

# -------------------------------------------------------------------------
# 3. MONTE CARLO RISK SIMULATION (10,000 Iterations)
# -------------------------------------------------------------------------


def run_monte_carlo_simulation(
    baseline_days=300,
    baseline_budget_m=20.0,
    n_iterations=10000,
    weather_multiplier=1.0,
    supply_shock=False,
):
    """Simulates critical path and budget volatility under stochastic disturbances."""
    # Beta-PERT style distribution for labor & subcontractor variability
    subcontractor_draws = np.random.beta(a=5, b=2, size=n_iterations)
    weather_draws = (
        np.random.poisson(lam=14 * weather_multiplier, size=n_iterations)
        * np.random.exponential(1.5, size=n_iterations)
    )

    supply_mult = (
        np.random.uniform(1.15, 1.35, size=n_iterations)
        if supply_shock
        else np.random.uniform(0.98, 1.08, size=n_iterations)
    )

    simulated_delays = (
        (weather_draws * 1.5)
        + ((1.0 - subcontractor_draws) * 0.25 * baseline_days)
        + np.random.normal(0, 8, size=n_iterations)
    )
    simulated_delays = np.maximum(0, simulated_delays)

    simulated_costs_m = (
        baseline_budget_m * (1.0 + (simulated_delays / baseline_days) * 0.30)
    ) * supply_mult

    # Value at Risk (VaR 95%) and Conditional VaR (CVaR 95%)
    var_delay_95 = np.percentile(simulated_delays, 95)
    cvar_delay_95 = simulated_delays[simulated_delays >= var_delay_95].mean()

    var_cost_95 = np.percentile(simulated_costs_m, 95)
    cvar_cost_95 = simulated_costs_m[simulated_costs_m >= var_cost_95].mean()

    return {
        "Mean_Delay_Days": round(float(np.mean(simulated_delays)), 1),
        "VaR_95_Delay_Days": round(float(var_delay_95), 1),
        "CVaR_95_Delay_Days": round(float(cvar_delay_95), 1),
        "Mean_Cost_M": round(float(np.mean(simulated_costs_m)), 2),
        "VaR_95_Cost_M": round(float(var_cost_95), 2),
        "CVaR_95_Cost_M": round(float(cvar_cost_95), 2),
    }


print("\n=== MONTE CARLO STRESS TEST RESULTS (10,000 RUNS) ===")
scenarios = {
    "Scenario A: Baseline Conditions": run_monte_carlo_simulation(),
    "Scenario B: Adverse Weather Shock (+35%)": run_monte_carlo_simulation(weather_multiplier=1.35),
    "Scenario C: Severe Supply & Tariff Disruption": run_monte_carlo_simulation(
        weather_multiplier=1.2, supply_shock=True
    ),
}
print(pd.DataFrame(scenarios).T.to_string())

[INFO] Ingested 600 project records. Positive class prevalence (Severe Risk): 41.33%


C:\Users\adity\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(



=== CLASSIFICATION BENCHMARK PERFORMANCE ===
                           Model  ROC-AUC  PR-AUC  F2-Score
Logistic Regression (ElasticNet)    0.925   0.915     0.752
        Random Forest Classifier    0.927   0.917     0.806
    Gradient Boosted Trees (GBM)    0.901   0.890     0.774

=== MONTE CARLO STRESS TEST RESULTS (10,000 RUNS) ===
                                               Mean_Delay_Days  VaR_95_Delay_Days  CVaR_95_Delay_Days  Mean_Cost_M  VaR_95_Cost_M  CVaR_95_Cost_M
Scenario A: Baseline Conditions                           53.0              123.4               161.6        21.68          23.38           24.17
Scenario B: Adverse Weather Shock (+35%)                  64.3              155.8               202.2        21.92          24.03           24.96
Scenario C: Severe Supply & Tariff Disruption             58.5              139.5               182.1        26.46          29.06           30.27
